# 03 · 函式與程式組織

把重複的邏輯包裝成函式（function），並學會用參數、作用域與型別提示，讓程式更清晰、可重用、好維護。

## 學習目標
- 能用 def 定義函式並以 return 回傳結果，理解有無回傳值的差異。
- 掌握位置參數（positional）、關鍵字參數（keyword）、預設值參數（default），以及 *args / **kwargs 的彈性傳參方式。
- 理解作用域（scope）、巢狀函式（nested function）與閉包（closure），並能用 nonlocal 修改外層變數。
- 能寫出純函式（pure function）與遞迴（recursion），判斷何時適用。
- 會用 lambda 匿名函式撰寫簡短函式，並用型別提示（type hints）標註參數與回傳型別。
- 認識模組（module）的三種 import 方式，能合理組織程式碼。

## 函式基礎

函式（function）是把一段可命名、可重複使用的邏輯封裝起來的基本單位。
想避免同樣的程式碼重複貼好幾次、想讓邏輯有個好懂的名字時就用它。

關鍵分辨：回傳值（return）把結果交回給呼叫端，可以再被運算或存起來；印出（print）只是顯示在畫面上，函式本身回傳的是 None。

形狀：
```
def 函式名(參數):
    ...
    return 結果
```

In [ ]:
# 概念：def 定義函式、return 回傳值，與「只 print 不 return」的差別
def c_to_f_return(celsius):
    return celsius * 9 / 5 + 32      # 把結果交回呼叫端，可再被使用

def c_to_f_print(celsius):
    print(celsius * 9 / 5 + 32)      # 只顯示在畫面上，沒有交回結果

# 呼叫（call）有 return 的版本：結果能存進變數、繼續算
f1 = c_to_f_return(25)
print('return 版拿到的值：', f1)
print('可繼續運算，再加 1：', f1 + 1)

# 注意：只 print 的版本，回傳的是 None，存起來沒有可用數值
f2 = c_to_f_print(25)               # 這行會印出 77.0，但 f2 拿到的是 None
print('print 版回傳的是：', f2)

## 參數的多種樣貌

傳參方式決定函式呼叫起來有多彈性。位置參數（positional argument）按順序對應；關鍵字參數（keyword argument）用名字對應、順序可變；預設值參數（default argument）讓某些參數可省略。

當參數數量不固定時，用 *args 收集任意數量的位置參數成 tuple，用 **kwargs 收集任意數量的關鍵字參數成 dict。

順序規則：一般參數 → 預設值參數 → *args → **kwargs。

In [ ]:
# 概念：位置參數、關鍵字參數、預設值參數，以及 *args / **kwargs
def greet(greeting, *names, punctuation='。', **options):
    # greeting 是必填位置參數；punctuation 有預設值可省略
    # *names 收集任意數量的稱呼成 tuple；**options 收集額外設定成 dict
    who = '、'.join(names) if names else '大家'   # 沒給名字就用「大家」
    line = f'{greeting}，{who}{punctuation}'
    for key, value in options.items():            # 把附加設定逐條附在後面
        line += f'（{key}={value}）'
    return line

# 只給必填參數：其餘走預設
print(greet('早安'))

# 多個稱呼進 *names，並用關鍵字參數覆寫標點
print(greet('哈囉', '小明', '小華', punctuation='！'))

# 技巧：**kwargs 讓呼叫端能丟入未事先列舉的設定
print(greet('您好', '王經理', mood='正式', lang='zh'))

## 純函式與函式設計觀念

純函式（pure function）只依賴傳入的參數、不改變外部狀態，輸入相同則輸出永遠相同。
相對地，會改到外部資料或環境的行為叫副作用（side effect）。

為什麼追求純函式：好測試、好重用、不會在別處偷偷改壞別人的資料，符合單一職責。學會辨識並避免不必要的副作用，是寫出可維護程式的關鍵。

In [ ]:
# 概念：對照「有副作用」與「純函式」兩種清單加總寫法
def add_value_impure(numbers, value):
    numbers.append(value)            # 注意：直接改到傳入的清單，是副作用
    return sum(numbers)

def add_value_pure(numbers, value):
    new_numbers = numbers + [value]  # 造一份新清單，不動原始資料
    return sum(new_numbers)

# 造一組假資料當示範用
data = [10, 20, 30]

# 呼叫有副作用版本：原始 data 被改掉了
total_impure = add_value_impure(data, 40)
print('impure 加總：', total_impure, '；呼叫後原始 data：', data)

# 重置後呼叫純函式版本：原始 data 維持不變
data = [10, 20, 30]
total_pure = add_value_pure(data, 40)
print('pure 加總：', total_pure, '；呼叫後原始 data：', data)

## 作用域、巢狀函式與閉包

作用域（scope）決定變數在哪裡看得到。函式內定義的是區域變數（local），外面看不到；函式外的是全域變數（global）。Python 查找變數時由內往外一層層找。

巢狀函式（nested function）是函式裡再定義函式。當內層函式記住並使用外層的變數，就形成閉包（closure）。預設情況下內層只能讀外層變數；要在內層修改外層變數，需用 nonlocal 宣告。

In [ ]:
# 概念：巢狀函式 + 閉包，用 nonlocal 在內層修改外層變數
def make_counter(start=0):
    count = start                    # 外層的區域變數，會被內層記住（形成閉包）

    def step():
        nonlocal count               # 注意：沒有 nonlocal，內層賦值會被當成新的區域變數
        count += 1                    # 累加外層那個 count
        return count

    return step                      # 回傳內層函式本身（連同它記住的 count）

# 每呼叫一次 make_counter 都會得到獨立的計數器
counter = make_counter()
print('第一次：', counter())
print('第二次：', counter())
print('第三次：', counter())

# 技巧：另開一個計數器互不干擾，證明各自記住自己的 count
other = make_counter(start=100)
print('另一個計數器：', other())

## lambda 與遞迴

lambda 是匿名函式（anonymous function），適合寫一行即用即丟的小函式，常拿來當排序的鍵（key）。

形狀：
```
lambda 參數: 運算式
```

遞迴（recursion）是函式呼叫自己，把問題拆成更小的同類問題。一定要設好終止條件（base case），否則會無限遞迴而崩潰。

In [ ]:
# 概念：lambda 當排序鍵、遞迴計算階乘與終止條件
words = ['banana', 'fig', 'cherry', 'kiwi']

# 用 lambda 取每個字串的長度當排序依據（即用即丟，不必另外 def）
by_length = sorted(words, key=lambda w: len(w))
print('依長度排序：', by_length)

def factorial(n):
    if n <= 1:                       # 終止條件：到 1 就停，避免無限遞迴
        return 1
    return n * factorial(n - 1)      # 把問題縮小一階，逐步收斂到終止條件

print('5 的階乘：', factorial(5))
print('逐步展開 3! =', '3 * 2 * 1 =', factorial(3))

## 型別提示

型別提示（type hints）在參數與回傳值上標註預期型別。它不影響程式執行，但能讓函式介面更清楚、利於編輯器與檢查工具找錯、也方便閱讀。

容器與特殊情況常用 typing 模組：List、Dict 標註容器內容，Optional[X] 表示可能是 X 也可能是 None（例如查不到資料），Any 表示任意型別。

形狀：
```
def 函式名(參數: 型別) -> 回傳型別:
```

In [ ]:
# 概念：用 type hints 標註參數與回傳型別，Optional 表示可能找不到
from typing import Optional, Dict, List

# 造一份假的使用者資料庫當示範用
users: List[Dict[str, str]] = [
    {'id': 'u01', 'name': '小明'},
    {'id': 'u02', 'name': '小華'},
]

def find_user(db: List[Dict[str, str]], user_id: str) -> Optional[Dict[str, str]]:
    # 回傳型別用 Optional：找到回傳 dict，找不到回傳 None
    for record in db:
        if record['id'] == user_id:
            return record
    return None

print('查到的使用者：', find_user(users, 'u02'))
print('查不到時回傳：', find_user(users, 'u99'))

# 技巧：型別提示只是註記，可用 __annotations__ 檢視，不會強制執行
print('函式的型別註記：', find_user.__annotations__)

## 模組與 import

模組（module）是一個 .py 檔，裡面裝著可重用的函式與變數。把程式拆成模組，能避免單檔過長、方便重用。

三種 import 寫法各有取捨：

| 寫法 | 呼叫方式 | 適用情境 |
|---|---|---|
| `import math` | `math.sqrt(x)` | 最清楚，明示來源，避免名稱衝突 |
| `import numpy as np` | `np.array(...)` | 模組名太長時取別名 |
| `from math import sqrt` | `sqrt(x)` | 只用少數幾個名字、想寫短一點 |

命名空間（namespace）就是「名字所屬的容器」，用 `math.` 這種前綴可避免不同模組的同名函式互相蓋掉。

In [ ]:
# 概念：三種 import 寫法的差異與呼叫名稱長短
import math                          # 寫法一：保留命名空間，math.xxx
import random as rnd                 # 寫法二：取別名，rnd.xxx
from math import pi                  # 寫法三：直接取名字進來，pi

print('math.sqrt(16) =', math.sqrt(16))      # 來源清楚
print('rnd 取別名後 seed 固定再取數：')
rnd.seed(0)                          # 固定亂數種子，讓示範輸出可重現
print('  random 值 =', round(rnd.random(), 4))
print('直接用 pi =', pi)

# 注意：from X import * 會把所有名字灌進來，易造成名稱衝突，正式程式少用
print('math.floor(pi) =', math.floor(pi))

## 練習

以下三題由淺入深，皆為複合型但簡短。資料請在題目內自己用 list 造（仿真即可），不要引用外部檔案。每題完成後對照「驗收」確認結果。

In [ ]:
# TODO 1 ·（簡單）樓地板面積計算函式（整合：函式基礎 + 型別提示）
#   情境：給定幾棟建築的占地面積（footprint area）與樓層數（floors），
#         要算出各棟的樓地板面積 GFA（gross floor area）。
#   要求：
#     1. 用 list 自己造 4 棟建築的資料：占地面積（平方公尺）與樓層數兩個清單（仿真數字即可）。
#     2. 定義函式 gfa(footprint: float, floors: int) -> float，加上型別提示，
#        以 return 回傳「占地面積 × 樓層數」。
#     3. 用迴圈呼叫該函式，把每棟的 GFA 收進一個 list。
#   驗收：應該看到 4 個 GFA 數值，且每個都等於對應的占地面積乘以樓層數。
# TODO: 在這裡完成

In [ ]:
# TODO 2 ·（中間）可設定的建築篩選統計（整合：參數的多種樣貌 + 純函式 + 型別提示 + 模組 import）
#   情境：一批建築資料含樓高（height）、樓層數（floors）與用途分類標籤（use），
#         要做可調條件的篩選與彙總。
#   要求：
#     1. 用 list 造一組建築資料（每棟是一個 dict，含 height、floors、use 三個鍵），數字仿真即可。
#     2. 定義純函式 summarize(buildings, min_height: float = 0.0, **filters)：
#        用預設值參數設高度門檻，用 **kwargs 接收額外等值篩選（例如 use='住宅'）；
#        函式不可修改傳入的原始資料。
#     3. 函式回傳一個 Dict（加型別提示），內含符合條件的棟數、平均樓高、最高樓層數。
#     4. 在主程式 import math（任一種 import 寫法），用它做四捨五入或開根號等其中一個收尾計算。
#   驗收：改變 min_height 或 **filters 後，回傳的棟數與平均值跟著變，且原始資料清單內容不變。
# TODO: 在這裡完成

In [ ]:
# TODO 3 ·（稍難）街廓網格的容積聚合與政策情境比較（整合：閉包 + nonlocal + 遞迴 / 聚合 + lambda + 純函式）
#   情境：把城市切成數個街廓網格 cell，每格內有若干建築，要聚合各格的總樓地板面積，
#         並比較「放寬容積率 FAR（floor area ratio）」政策前後的變化。
#   要求：
#     1. 用巢狀 list 造資料：數個 cell，每個 cell 是一串建築的 (footprint, floors)（仿真數字即可）。
#     2. 寫一個外層函式回傳閉包累加器，內層用 nonlocal 累加跑過的總 GFA；
#        用它走訪所有 cell 聚合出全市總量（可用遞迴或迴圈走訪巢狀結構）。
#     3. 用 lambda 定義政策乘數（例如 apply = lambda gfa, k: gfa * k），
#        以純函式方式對每格 GFA 套用放寬倍率 k，產生情境後的新值（不改原始資料）。
#     4. 比較政策前後的全市總 GFA，並找出增量最大的那一格。
#   驗收：政策後總量大於政策前，且能正確指出增幅最大的 cell 編號。
# TODO: 在這裡完成

## 小結

- 函式用 def 定義、用 return 把結果交回呼叫端；return 與 print 是兩回事，沒有 return 時回傳 None。
- 參數有位置、關鍵字、預設值三種傳法，*args / **kwargs 收集不定量參數，順序規則固定。
- 純函式只依賴參數、不改外部狀態，較好測試與重用；要警覺並避免不必要的副作用。
- 內層函式可記住外層變數形成閉包，nonlocal 讓內層能修改外層變數。
- lambda 寫一行即用即丟的小函式；遞迴一定要設終止條件才會收斂。
- 型別提示讓介面更清楚但不影響執行；用 Optional / List / Dict 標註特殊與容器型別。
- 三種 import 寫法各有取捨，命名空間能避免名稱衝突，模組讓程式好拆好重用。